# Program 8 – Hyperparameter Tuning using Keras Tuner

**Objective:** Tune hyperparameters (units, learning rate, activation) using Keras Tuner.

**Dataset:** MNIST

**Key tasks**
1. Install `keras-tuner`.
2. Define a model-building function.
3. Search best parameters using RandomSearch (and Hyperband).
4. Display the best configuration and final accuracy.

## Theory (for viva)

- **Parameters** (weights, biases) are learned by gradient descent.
- **Hyperparameters** are chosen *before* training: number of units, number of layers, activation, learning rate, dropout, batch size ...
- **Keras Tuner** automates searching for good hyperparameters.

**Search space** is defined with the `hp` object inside a model-building function:
| Method | Example | Meaning |
|---|---|---|
| `hp.Int` | `hp.Int("units", 32, 512, step=32)` | integer in a range |
| `hp.Choice` | `hp.Choice("activation", ["relu", "tanh"])` | pick one from a list |
| `hp.Float` | `hp.Float("lr", 1e-4, 1e-2, sampling="log")` | real number (log scale good for learning rates) |

**Tuners**
- **RandomSearch:** tries `max_trials` random combinations. Simple, surprisingly effective.
- **Hyperband:** trains many models for few epochs, keeps the best ones and trains them longer (successive halving). Faster for large search spaces.
- **BayesianOptimization:** uses results of previous trials to choose the next combination.

The tuner picks the trial with the best `objective` (here `val_accuracy`).

## Step 1 – Install keras-tuner

In [ ]:
!pip install -q keras-tuner   # Install Keras Tuner quietly (only needed once per environment)

## Step 2 – Import libraries

In [ ]:
import keras  # Keras 3 API (runs on TensorFlow backend)
from keras import layers  # Layer classes (Dense, Dropout, Embedding, ...)
import keras_tuner as kt  # Keras Tuner for hyperparameter search
import matplotlib.pyplot as plt  # Plotting graphs
import numpy as np  # Numerical arrays
import tensorflow as tf  # TensorFlow (tensors, GradientTape, seeds)

print("Keras Tuner version:", kt.__version__)   # Confirm installation
tf.random.set_seed(42)                  # Reproducibility

## Step 3 – Load and normalize MNIST

In [ ]:
(x_train, y_train), (x_test, y_test) = keras.datasets.mnist.load_data()  # Load MNIST
x_train = x_train.astype("float32") / 255.0                             # Scale train pixels to [0, 1]
x_test = x_test.astype("float32") / 255.0                               # Scale test pixels to [0, 1]

x_val, y_val = x_train[-10000:], y_train[-10000:]                       # Last 10,000 training images for validation
x_train_sub, y_train_sub = x_train[:-10000], y_train[:-10000]           # Remaining 50,000 for training during search

print("Train:", x_train_sub.shape, "Val:", x_val.shape, "Test:", x_test.shape)  # Check split sizes

## Step 4 – Define the model-building function (search space)

In [ ]:
def build_model(hp):                                                    # Keras Tuner calls this with an `hp` object
    model = keras.Sequential()                                          # Empty sequential model
    model.add(keras.Input(shape=(28, 28)))                              # Input: 28x28 image
    model.add(layers.Flatten())                                         # Flatten to 784 vector

    activation = hp.Choice("activation", ["relu", "tanh", "elu"])       # Tune activation function

    for i in range(hp.Int("num_layers", 1, 3)):                         # Tune number of hidden layers: 1, 2 or 3
        model.add(layers.Dense(                                         # Add a hidden Dense layer
            units=hp.Int(f"units_{i}", min_value=32, max_value=512, step=32),  # Tune neurons per layer: 32..512
            activation=activation,                                      # Use tuned activation
        ))
        model.add(layers.Dropout(hp.Float(f"dropout_{i}", 0.0, 0.5, step=0.1)))  # Tune dropout rate 0.0..0.5

    model.add(layers.Dense(10, activation="softmax"))                   # Output layer: 10 digit probabilities

    learning_rate = hp.Float("learning_rate", 1e-4, 1e-2, sampling="log")  # Tune learning rate on log scale
    model.compile(                                                      # Compile the model
        optimizer=keras.optimizers.Adam(learning_rate=learning_rate),   # Adam with tuned learning rate
        loss="sparse_categorical_crossentropy",                         # Integer labels
        metrics=["accuracy"],                                           # Accuracy metric (val_accuracy used as objective)
    )
    return model                                                        # Return compiled model

build_model(kt.HyperParameters()).summary()                             # Build once with default values to test the function

## Step 5 – Create the RandomSearch tuner

In [ ]:
tuner = kt.RandomSearch(                                                # Random search tuner
    hypermodel=build_model,                                             # Function that builds models
    objective="val_accuracy",                                           # Pick trial with highest validation accuracy
    max_trials=10,                                                      # Try 10 different hyperparameter combinations
    executions_per_trial=1,                                             # Train each combination once
    directory="kt_logs",                                                # Folder where results are saved
    project_name="mnist_random_search",                                 # Sub-folder name
    overwrite=True,                                                     # Start fresh (delete old results)
)

tuner.search_space_summary()                                            # Print all hyperparameters and their ranges

## Step 6 – Run the search

In [ ]:
stop_early = keras.callbacks.EarlyStopping(monitor="val_loss", patience=2)  # Stop a trial early if val_loss stops improving

tuner.search(                                                           # Start hyperparameter search
    x_train_sub, y_train_sub,                                           # Training data
    epochs=5,                                                           # Max epochs per trial (short to save time)
    batch_size=128,                                                     # Batch size
    validation_data=(x_val, y_val),                                     # Validation data used for the objective
    callbacks=[stop_early],                                             # Early stopping
    verbose=1,                                                          # Show progress
)

## Step 7 – Display results and best configuration

In [ ]:
tuner.results_summary(num_trials=5)                                     # Print top-5 trials with their scores

best_hp = tuner.get_best_hyperparameters(num_trials=1)[0]               # Best hyperparameter set

print("\n===== BEST HYPERPARAMETERS =====")                             # Header
print("Activation    :", best_hp.get("activation"))                     # Best activation
print("Num layers    :", best_hp.get("num_layers"))                     # Best number of hidden layers
for i in range(best_hp.get("num_layers")):                              # For each hidden layer used
    print(f"  Layer {i}: units = {best_hp.get(f'units_{i}')}, "
          f"dropout = {best_hp.get(f'dropout_{i}'):.1f}")               # Units and dropout of layer i
print("Learning rate :", f"{best_hp.get('learning_rate'):.5f}")         # Best learning rate

## Step 8 – Retrain the best model on full training data and evaluate

In [ ]:
best_model = tuner.hypermodel.build(best_hp)                            # Build a fresh model with best hyperparameters

history = best_model.fit(                                               # Train longer than during search
    x_train, y_train,                                                   # Full 60,000 training images
    epochs=15,                                                          # More epochs for final model
    batch_size=128,                                                     # Batch size
    validation_split=0.1,                                               # 10% for validation curves
    callbacks=[keras.callbacks.EarlyStopping(monitor="val_loss",        # Early stopping on val loss
                                             patience=3,                # Wait 3 epochs
                                             restore_best_weights=True)],  # Keep best weights
    verbose=2,                                                          # One line per epoch
)

test_loss, test_acc = best_model.evaluate(x_test, y_test, verbose=0)    # Evaluate on test set
print(f"\nFINAL TEST ACCURACY with best hyperparameters: {test_acc:.4f}")  # Final answer

In [ ]:
plt.figure(figsize=(12, 4))                                             # Figure
plt.subplot(1, 2, 1)                                                    # Left plot
plt.plot(history.history["loss"], label="Train")                        # Training loss
plt.plot(history.history["val_loss"], label="Validation")               # Validation loss
plt.title("Best model – Loss")                                          # Title
plt.xlabel("Epoch")                                                     # X label
plt.legend()                                                            # Legend
plt.subplot(1, 2, 2)                                                    # Right plot
plt.plot(history.history["accuracy"], label="Train")                    # Training accuracy
plt.plot(history.history["val_accuracy"], label="Validation")           # Validation accuracy
plt.title("Best model – Accuracy")                                      # Title
plt.xlabel("Epoch")                                                     # X label
plt.legend()                                                            # Legend
plt.tight_layout()                                                      # Tidy
plt.show()                                                              # Show

## Step 9 – Compare all trials visually

In [ ]:
trials = tuner.oracle.get_best_trials(num_trials=10)                    # All finished trials, best first
scores = [t.score for t in trials]                                      # Validation accuracy of each trial
labels = [f"{t.hyperparameters.get('activation')}\n"                    # Activation name
          f"lr={t.hyperparameters.get('learning_rate'):.4f}"            # Learning rate
          for t in trials]                                              # One label per trial

plt.figure(figsize=(14, 5))                                             # Figure
positions = np.arange(len(scores))                                      # One x position per trial
plt.bar(positions, scores, color="tab:purple")                          # Bar per trial
plt.xticks(positions, labels, fontsize=8)                               # Trial labels on x-axis
plt.ylim(min(scores) - 0.01, 1.0)                                       # Zoom y-axis
plt.ylabel("Validation accuracy")                                       # Y label
plt.title("RandomSearch trials (sorted best → worst)")                  # Title
plt.show()                                                              # Show

## Step 10 (optional) – Same search with Hyperband

In [ ]:
hb_tuner = kt.Hyperband(                                                # Hyperband tuner
    hypermodel=build_model,                                             # Same model-building function
    objective="val_accuracy",                                           # Same objective
    max_epochs=10,                                                      # Max epochs a model can be trained in one bracket
    factor=3,                                                           # Keep top 1/3 of models at each round
    directory="kt_logs",                                                # Results folder
    project_name="mnist_hyperband",                                     # Separate sub-folder
    overwrite=True,                                                     # Start fresh
)

hb_tuner.search(x_train_sub, y_train_sub,                               # Run Hyperband search
                validation_data=(x_val, y_val),                         # Validation data
                callbacks=[stop_early],                                 # Early stopping
                verbose=1)                                              # Show progress

hb_best = hb_tuner.get_best_hyperparameters(1)[0]                       # Best hyperparameters from Hyperband
print("Hyperband best:", hb_best.values)                                # Print them as a dict

## Expected observations
- Best trials usually use **relu/elu**, 1–3 layers of 256–512 units, learning rate ≈ 1e-3.
- Final test accuracy ≈ **98%**.
- Very high learning rates (≈1e-2) or tiny layers give the worst trials.

## Viva questions
1. Parameter vs hyperparameter? → Parameters are learned (weights); hyperparameters are set before training (lr, units...).
2. Why sample learning rate on a log scale? → Useful values span orders of magnitude (1e-4 … 1e-2).
3. Why use a separate validation set during search? → Choosing hyperparameters on test data would leak test information (overfitting to test set).
4. RandomSearch vs GridSearch? → Grid tries all combinations (expensive); random samples combinations and often finds good ones faster.
5. How does Hyperband save time? → Stops poor configurations early and gives more epochs only to promising ones.